<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/cursor/long-run-notebook-067c/notebooks/09_long_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Long run: one BandTransformer schedule across many Colab sessions, shared by 2 to 4 people

The screens (`08`) ran 30 epochs. This notebook trains for hundreds to thousands of epochs, so the model gets a real chance to learn, and it survives the 12-hour Colab limit: a stop is clean, and the next session resumes from the checkpoint on Drive. **Runtime → Change runtime type → GPU, set `RUNNER_NAME` in the first code cell, then Runtime → Run all.**

Screen results to beat (val FG mIoU, 30 epochs): fused transformer 0.286, collapsed control 0.298, CNN 0.403.

| Step | What | Stops when |
|---|---|---|
| Settings | who you are, which run, the recipe, the epochs target, the time budget | a name or a flag is invalid |
| Setup | mount Drive, clone `BRANCH`, install, check the GPU | no GPU, or the branch does not take the flags |
| Data | reuse `/content/data`, or `scripts/get_data.sh` | md5 or split counts wrong |
| Smoke | two steps of the model | the parameter line is wrong |
| Launch | claim `ROOT/<RUN_TAG>`, optional fork, start `python -m sw.longrun run` in the background | another live session holds the tag |
| Monitor | progress, heartbeat, stall check | the run died, hung, or its log shows the wrong data or model |
| Leaderboard | every run under `ROOT`: epoch, best val FG mIoU, owner, state | |
| Plots | train loss, train and val FG mIoU, per-class IoU at best, all runs together | |
| Soup | optional, experimental: average forks of one parent | |

**Schedule.** Cosine over `EPOCHS_TARGET` epochs after a linear warmup, `--lr 3e-4` for the transformers (the CNN keeps `1e-3`), `--standardize`, gradient clipping, a cosine floor of 1 % of the peak rate. Early stopping is off. The epochs target is fixed when a run starts: a resume keeps the saved schedule. To train longer, **fork** the run (below) instead of raising the target.

**Sessions.** Checkpoints go to `ROOT/<RUN_TAG>/` on Drive every `SAVE_EVERY` epochs (atomic write; `last.pt` plus the newest `KEEP_LAST` copies plus `best.pt`). Validation runs every `EVAL_EVERY` epochs. `TIME_BUDGET_MINUTES` after the run starts it stops after a saved epoch; run all again in a new session and it resumes with the model, optimizer, scheduler, epoch, random state and logs. After a disconnect, the old session's lock goes stale in 10 minutes; this notebook waits for that if the lock is yours.

**Teams.** All runs live under one shared Drive folder (`docs/COLAB.md`, "Long runs"). Each run has a lock (owner, session start, heartbeat), a `metadata.json` (git sha, flags, seed, owner, data fingerprint, parent), and shows up in the leaderboard. Each person keeps their own `/content/data`. Training is one run on one VM; this does **not** split one run across several Colab VMs.

AI-assisted notebook (Cursor cloud agent); Scott reviews every line.

## Settings

Edit here, or set the `LONG_*` environment variables (the local test does that). `--epochs` is passed once, inside `TRAIN_ARGS`. No run takes `--stats-path`: each run folder keeps its own band statistics, and its `metadata.json` records a fingerprint of the data they came from.

In [ ]:
import os
import re
from pathlib import Path


def _env(name, default):
    return os.environ.get(name, default)


# ---- who and which run ----
RUNNER_NAME = _env("LONG_RUNNER", "")  # your name: on the leaderboard and in the lock. Required.
MODEL = _env("LONG_MODEL", "fused")  # "fused" (BandTransformer 2+6), "collapsed" (the 0+8 control) or "cnn"
RUN_TAG = _env("LONG_TAG", "")  # one folder per run under ROOT. Empty: "<runner>-<model>"
AUTO_NEW_TAG = False  # True: if a live session of someone else holds RUN_TAG, take "<tag>-2" instead of stopping
FORCE_TAKEOVER = False  # True: take the tag over even with a live heartbeat. Only for your own session that just lost its VM.

# ---- the schedule ----
EPOCHS_TARGET = int(_env("LONG_EPOCHS", "1000"))  # fixed for the run: cosine decays to the floor at this epoch
LR = float(_env("LONG_LR", "0")) or None  # None: 3e-4 for the transformers, 1e-3 for the CNN. Always passed to sw.train explicitly.
SEED = int(_env("LONG_SEED", "0"))
DOWNSAMPLE = int(_env("LONG_DOWNSAMPLE", "2"))  # half resolution, as in the screens
BATCH_SIZE = int(_env("LONG_BATCH", "4"))
AUGMENT = True  # False passes --no-aug
WARMUP_EPOCHS = int(_env("LONG_WARMUP_EPOCHS", "5"))  # linear warmup; a fork that takes the optimizer usually wants 0
CLIP_GRAD_NORM = 1.0  # None: no clipping
COSINE_FLOOR = 0.01  # the rate never falls below LR x this
PE_INIT_STD = None  # optional: 0.02 shrinks the band position embedding at init (transformers with band tokens)
AMP = False  # True: mixed precision on the GPU (not measured here)
TRAIN_IMAGES = 514  # official train split; only used to turn WARMUP_EPOCHS into optimizer steps

# ---- checkpoints and time ----
SAVE_EVERY = int(_env("LONG_SAVE_EVERY", "5"))  # epochs between writes of last.pt
KEEP_LAST = int(_env("LONG_KEEP_LAST", "3"))  # also keep this many ckpt_eNNNNN.pt copies (forks start from these)
EVAL_EVERY = int(_env("LONG_EVAL_EVERY", "5"))  # epochs between full-val FG mIoU and per-class IoU
TIME_BUDGET_MINUTES = float(_env("LONG_BUDGET", "600"))  # stop cleanly after a saved epoch before this; Colab ends sessions near 12 h

# ---- fork: start a NEW tag from another run's checkpoint ----
FORK_FROM = _env("LONG_FORK_FROM", "") or None  # the parent's tag, e.g. "ann-fused"
FORK_EPOCH = int(_env("LONG_FORK_EPOCH", "0")) or None  # None: its last.pt. An epoch needs KEEP_LAST > 0 in the parent.
FORK_OPTIMIZER = False  # True: also take the AdamW moments (the run's own lr and weight decay still apply)

# ---- comparison and soup ----
COMPARE_TAGS = []  # other runs to put next to this one in the per-class plot, e.g. ["tim-cnn"]
SOUP_RUNS = [t for t in _env("LONG_SOUP", "").split(",") if t]  # EXPERIMENTAL: forks of one parent to average, e.g. ["ann-a", "bob-b"]
SOUP_TAG = None  # the soup is saved under this tag only if it beats every ingredient

# ---- where ----
BRANCH = _env("LONG_BRANCH", "cursor/long-run-notebook-067c")  # after the PR is merged: wp7-screen-notebook
REPO_URL = "https://github.com/scottn66/255-final-project.git"
ROOT = Path(_env("LONG_ROOT", "/content/drive/MyDrive/SW_runs/long_run"))  # the SHARED folder (a Drive shortcut named SW_runs)
WORK = Path(_env("LONG_WORK", "/content/sw_long"))  # local disk: the smoke run
DATA_DIR = Path(_env("LONG_DATA", "/content/data"))  # local disk: the cubes (never train from Drive)
LONG_ZIP = _env("LONG_ZIP", "")  # optional: a copy of spectralwaste_segmentation.zip on Drive
DELETE_ZIP = True
DEVICE = _env("LONG_DEVICE", "cuda")
NUM_WORKERS = int(_env("LONG_WORKERS", "2"))
POLL_SECONDS = int(_env("LONG_POLL", "300"))
STALL_FLOOR_SECONDS = int(_env("LONG_STALL_FLOOR", "3600"))  # a live run is stuck after max(this, 5 x its median epoch time) with no log growth
STALL_EPOCHS = 5

# ---- checks and the flags ----
NAME_OK = re.compile(r"[A-Za-z0-9][A-Za-z0-9_.-]*")
ARCH = {
    "fused": ["--model", "transformer"],
    "collapsed": ["--model", "transformer", "--tokenizer", "collapsed", "--depth-spectral", "0", "--depth-spatial", "8"],
    "cnn": ["--model", "cnn"],
}
PARAM_LINE = {
    "fused": "BandTransformer parameters: 488,487 (0.488M)",
    "collapsed": "BandTransformer parameters: 491,191 (0.491M)",
    "cnn": "MiniNetSeg parameters: 571,879 (0.572M)",
}
if MODEL not in ARCH:
    raise ValueError(f"MODEL must be one of {list(ARCH)}; got {MODEL!r}")
if not NAME_OK.fullmatch(RUNNER_NAME):
    raise ValueError("Set RUNNER_NAME (letters, digits, - _ .): it is on the leaderboard and in the lock of every run you start")
if not RUN_TAG:
    RUN_TAG = f"{RUNNER_NAME.lower()}-{MODEL}" + (f"-from-{FORK_FROM}" if FORK_FROM else "")
for _name in (RUN_TAG, FORK_FROM, SOUP_TAG, *COMPARE_TAGS, *SOUP_RUNS):
    if _name is not None and not NAME_OK.fullmatch(_name):
        raise ValueError(f"{_name!r} is not a valid run tag (letters, digits, - _ .)")
if FORK_FROM == RUN_TAG:
    raise ValueError("a fork needs its own RUN_TAG, not its parent's")
if FORK_OPTIMIZER and not FORK_FROM:
    raise ValueError("FORK_OPTIMIZER needs FORK_FROM")
if EPOCHS_TARGET < 1 or SAVE_EVERY < 1 or EVAL_EVERY < 1 or KEEP_LAST < 0 or TIME_BUDGET_MINUTES <= 0:
    raise ValueError("EPOCHS_TARGET, SAVE_EVERY and EVAL_EVERY must be >= 1, KEEP_LAST >= 0, TIME_BUDGET_MINUTES > 0")
if "input_blind_ab" in ROOT.parts:
    raise ValueError(f"{ROOT}: that folder holds band statistics made from 64 images; use another runs folder")

if LR is None:
    LR = 1e-3 if MODEL == "cnn" else 3e-4
WARMUP_STEPS = WARMUP_EPOCHS * (TRAIN_IMAGES // BATCH_SIZE)

# Everything sw.train gets, except --data-root (added at launch) and the fork flags. --epochs appears here once.
TRAIN_ARGS = [
    *ARCH[MODEL],
    "--lr", str(LR),
    "--epochs", str(EPOCHS_TARGET),
    "--seed", str(SEED),
    "--downsample", str(DOWNSAMPLE),
    "--batch-size", str(BATCH_SIZE),
    "--standardize",
    "--warmup-steps", str(WARMUP_STEPS),
    "--cosine-floor", str(COSINE_FLOOR),
    "--save-every", str(SAVE_EVERY),
    "--keep-last", str(KEEP_LAST),
    "--eval-every", str(EVAL_EVERY),
    "--max-minutes", str(TIME_BUDGET_MINUTES),
    "--num-workers", str(NUM_WORKERS),
    "--device", DEVICE,
]
if CLIP_GRAD_NORM is not None:
    TRAIN_ARGS += ["--clip-grad-norm", str(CLIP_GRAD_NORM)]
if PE_INIT_STD is not None:
    TRAIN_ARGS += ["--pe-init-std", str(PE_INIT_STD)]
if not AUGMENT:
    TRAIN_ARGS += ["--no-aug"]
if AMP:
    TRAIN_ARGS += ["--amp"]

## Setup

On Colab: mount Drive, clone `BRANCH` (or pull it if the clone is there), install `requirements.txt`, check for a GPU, and check that this branch's `sw.train` takes every flag above. Locally: use the repo this notebook is in.

`ROOT` is the shared folder. In Drive, each teammate adds a shortcut to the shared folder into **My Drive** with the same name (`SW_runs`); then `ROOT` is the same folder for everyone. A `ROOT` that does not exist is created, privately, and the cell says so.

In [ ]:
import contextlib
import io
import json
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    """Run a command, show its output, raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


def tail(path, n=25):
    return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:]) if Path(path).is_file() else "(no log)"


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(
                f"{REPO} has local changes that block switching to {BRANCH}; delete the folder and Run all again"
            ) from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env_root = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env_root) if _env_root else next(
        p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file()
    )

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch
from sw import gates, longrun

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
ON_BRANCH = subprocess.run(["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True).stdout.strip()

# The flags must parse on this branch before any data is touched.
_err = io.StringIO()
try:
    from sw.train import _build_parser

    with contextlib.redirect_stderr(_err):
        _build_parser().parse_args(TRAIN_ARGS)
except (SystemExit, ImportError) as err:
    _why = (_err.getvalue().strip().splitlines() or [str(err)])[-1]
    raise RuntimeError(
        f"{REPO} (branch {ON_BRANCH or BRANCH}, commit {COMMIT}) does not take these flags: {_why}\n"
        "This notebook needs --save-every, --keep-last, --eval-every, --max-minutes and --init-from. Set BRANCH to cursor/long-run-notebook-067c or a branch with it."
    ) from None

if not ROOT.is_dir():
    print(f"NOTE: {ROOT} did not exist and is created now. If you meant the team's shared folder, add its shortcut to My Drive first.")
ROOT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE

print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"runner {RUNNER_NAME}  run {RUN_TAG}  model {MODEL}  epochs target {EPOCHS_TARGET}  lr {LR}")
print(f"root   {ROOT}\nwork   {WORK}\ndata   {DATA_DIR}")

## Data

Each person keeps their own copy on local disk; nothing here writes the cubes to Drive. In this order:

1. `SW_ROOT` is set and already holds the full train and val splits. Nothing is downloaded.
2. `LONG_ZIP` points at a copy of the zip on Drive. It is copied to local disk, then checked and unpacked.
3. `scripts/get_data.sh` downloads the zip (resumable), checks its md5, and unzips it.

The counts must be 514 train and 167 val. The official split only. The monitor checks the same two numbers in the run's log, and `metadata.json` stores a fingerprint of the file names, so a run cannot silently continue on other data.

In [ ]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def split_sizes(root):
    return {s: len(SpectralWasteDataset(root, split=s, random_flip=False)) for s in ("train", "val")}


def counts_ok(root):
    try:
        return all(n == SPLIT_COUNTS[s] for s, n in split_sizes(root).items())
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("LONG_ALLOW_PARTIAL") == "1"  # local test only: a small SW_ROOT
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test; not the official split)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if LONG_ZIP and Path(LONG_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", LONG_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)

SIZES = split_sizes(SW_ROOT)
IMAGES_LINE = f"train images {SIZES['train']}  val images {SIZES['val']}  "
DATA_FP = longrun.data_fingerprint(SW_ROOT)
print(f"every run must log: {IMAGES_LINE.strip()}")
print(f"data fingerprint {DATA_FP}")

## Smoke

Two optimizer steps of the model with `--standardize`, before any long run. It must print its parameter line (`488,487` fused, `491,191` collapsed, `571,879` CNN). A wrong line stops the notebook.

In [ ]:
smoke = subprocess.run(
    [
        sys.executable, "-m", "sw.train", *ARCH[MODEL], "--smoke", "--standardize",
        "--device", DEVICE, "--data-root", str(SW_ROOT), "--out-dir", str(WORK / f"smoke_{MODEL}"),
    ],
    capture_output=True, text=True, env=ENV, cwd=REPO,
)
print(smoke.stdout[-1500:], smoke.stderr[-3000:], sep="\n")
if smoke.returncode != 0:
    raise RuntimeError("smoke failed: the run raised (traceback above)")
if PARAM_LINE[MODEL] not in smoke.stdout:
    raise RuntimeError(f"smoke failed: the output has no line '{PARAM_LINE[MODEL]}'")
print("smoke passed for", MODEL)

## Launch

Claims `ROOT/<RUN_TAG>`, then starts `python -m sw.longrun run` in the background. That process holds the lock: it writes a heartbeat every minute, so closing this tab or interrupting a cell does not make a live run look dead, and a dead VM goes stale 10 minutes after its last beat.

* **New tag:** starts at epoch 1. **Existing tag:** resumes from `last.pt` (model, optimizer, scheduler, epoch, random state; the logs are trimmed to that epoch). A finished tag is left alone.
* **Someone else's live heartbeat:** this cell stops (or takes `<tag>-2` with `AUTO_NEW_TAG`). **Your own live lock from another session:** it waits for the heartbeat to go stale, up to 11 minutes, then takes over.
* **Fork:** with `FORK_FROM`, a new tag starts from another run's checkpoint (`FORK_EPOCH`, or its `last.pt`) with this notebook's lr, seed and augmentation. The parent and epoch go into `metadata.json`. A fork starts a fresh schedule of `EPOCHS_TARGET` epochs.
* **Guards:** the epochs target of an existing run cannot change (the cosine schedule is fixed; fork to train longer), and a folder started on other data or flags is refused.

In [ ]:
RUN_DIR = ROOT / RUN_TAG
PROC = None
SESSION = None


def check_target(run_dir):
    meta = longrun.read_json(run_dir / longrun.META_FILE) or {}
    target = meta.get("epochs_target")
    if target is not None and target != EPOCHS_TARGET:
        raise RuntimeError(
            f"{run_dir.name} was started with EPOCHS_TARGET {target}, not {EPOCHS_TARGET}. A resume keeps the saved cosine schedule, "
            "so changing the target would make the rate climb again. Set it back, or fork this run into a new RUN_TAG to train longer."
        )


def start(tag):
    run_dir = ROOT / tag
    check_target(run_dir)
    args = [*TRAIN_ARGS, "--data-root", str(SW_ROOT)]
    if FORK_FROM:
        forked = longrun.fork(ROOT / FORK_FROM, run_dir, TRAIN_ARGS, epoch=FORK_EPOCH, optimizer=FORK_OPTIMIZER)
        args += forked["args"]
        print(f"fork of {forked['parent']['tag']} at epoch {forked['parent']['epoch']}"
              f"{' with its optimizer state' if forked['parent']['optimizer'] else ''}")
    return longrun.launch(run_dir, RUNNER_NAME, args, REPO, SW_ROOT, env=ENV, force=FORCE_TAKEOVER)


def start_when_free(tag):
    """start(tag); wait for my own stale lock; on someone else's live lock stop, or take a new tag."""
    global RUN_TAG, RUN_DIR
    give_up = time.time() + longrun.STALE_SECONDS + 90
    while True:
        try:
            return start(tag)
        except longrun.LockHeld as err:
            info = longrun.run_status(ROOT / tag)
            lock = info["lock"] or {}
            if lock.get("owner") == RUNNER_NAME and info["state"] == "running" and time.time() < give_up:
                print(f"{tag}: your own session {lock.get('session')} has the lock (heartbeat {info['age']:.0f} s ago); "
                      "waiting for it to go stale if that VM is gone", flush=True)
                time.sleep(30)
                continue
            if not AUTO_NEW_TAG:
                raise
            tag = longrun.free_tag(ROOT, tag)
            print(f"{err}\n-> taking {tag} instead")
            RUN_TAG, RUN_DIR = tag, ROOT / tag


PROC, SESSION = start_when_free(RUN_TAG)
RUN_DIR = ROOT / RUN_TAG
state = longrun.run_status(RUN_DIR)["state"]
meta = longrun.read_json(RUN_DIR / longrun.META_FILE) or {}
print(f"{RUN_TAG}: {state}  session {SESSION}  owner {RUNNER_NAME}  sha {meta.get('git_sha', '')[:7]}  parent {meta.get('parent')}")
if state == "done":
    print("nothing to train: this run finished. Fork it into a new RUN_TAG to train longer.")

## Monitor

Prints progress every `POLL_SECONDS`. Interrupt it any time: the run keeps going, and re-running this cell picks it up. It ends in one of these ways.

* **Done:** the run reached `EPOCHS_TARGET` and wrote `DONE.json`.
* **Paused:** the time budget stopped it after a saved epoch. Run all again in a new session.
* **Died:** raises, with the end of `train.log`. This cell never waits on a dead run (the notebook 07 hang). A process that exited without `DONE.json` or a final lock, a dead pid, a lock whose heartbeat went stale, or a `failed` lock all count.
* **Stuck:** the process is alive but `train.log` and `metrics.jsonl` have not grown for `max(STALL_FLOOR_SECONDS, STALL_EPOCHS x median epoch time)`. Raises and prints the pid; nothing is killed.
* **Wrong run:** `train.log` must show the image counts from the Data cell and this model's parameter line.

In [ ]:
GROWTH = {}


def quiet_seconds():
    sizes = tuple(
        (RUN_DIR / name).stat().st_size if (RUN_DIR / name).is_file() else -1 for name in ("train.log", "metrics.jsonl")
    )
    if not GROWTH or GROWTH["sizes"] != sizes:
        GROWTH.update(sizes=sizes, since=time.time())
    return time.time() - GROWTH["since"]


def rows_now():
    try:
        return gates.read_metrics(RUN_DIR / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return []


def stall_limit():
    sec = gates.median_epoch_seconds(rows_now()) or 0.0
    return max(STALL_FLOOR_SECONDS, STALL_EPOCHS * sec)


def log_wrong():
    log = RUN_DIR / "train.log"
    lines = log.read_text(errors="replace").splitlines() if log.is_file() else []
    wrong = [ln for ln in lines if ln.startswith("train images ") and not ln.startswith(IMAGES_LINE)]
    wrong += [ln for ln in lines if re.match(r"\w+ parameters: ", ln) and ln != PARAM_LINE[MODEL]]
    return wrong


def line():
    info = longrun.run_status(RUN_DIR)
    rows = rows_now()
    if not rows:
        return f"{RUN_TAG:24s} {info['state']:8s} no epoch finished yet"
    scored = [r for r in rows if r.get("val_fg_miou") is not None]
    best = max(scored, key=lambda r: r["val_fg_miou"]) if scored else None
    sec = gates.median_epoch_seconds(rows) or 0.0
    left = sec * max(0, EPOCHS_TARGET - int(rows[-1]["epoch"])) / 3600
    beat = "" if info["age"] is None else f"  heartbeat {info['age']:.0f} s ago"
    best_txt = "n/a" if best is None else f"{best['val_fg_miou']:.3f} at {best['epoch']}"
    last_val = scored[-1]["val_fg_miou"] if scored else None
    return (
        f"{RUN_TAG:24s} {info['state']:8s} epoch {rows[-1]['epoch']}/{EPOCHS_TARGET}  loss {rows[-1]['train_loss']:.4f}  "
        f"val FG {'n/a' if last_val is None else format(last_val, '.3f')} (best {best_txt})  {sec:.1f} s/epoch  about {left:.1f} h left{beat}"
    )


def is_dead():
    """True when no process is training this folder. Do not call it while a finish is being written."""
    if PROC is not None:
        return PROC.poll() is not None
    return gates.run_state(RUN_DIR) in ("new", "partial")


def explain(reason):
    print(f"\n{RUN_TAG}: {reason}. End of its log:\n{tail(RUN_DIR / 'train.log')}\n", flush=True)
    raise RuntimeError(
        f"{RUN_TAG}: {reason}. Fix the cause, then Run all again: the run resumes from its last.pt (log tail above)."
    )


while True:
    print(time.strftime("%H:%M:%S"), line(), flush=True)
    wrong = log_wrong()
    if wrong:
        shown = "\n  ".join(wrong)
        raise RuntimeError(
            f"{RUN_TAG}: {RUN_DIR / 'train.log'} shows the wrong run:\n  {shown}\n"
            f"expected '{IMAGES_LINE.strip()}' and '{PARAM_LINE[MODEL]}'. Stop it (pid in {RUN_DIR / gates.RUNNING_FILE}), "
            "then use a new RUN_TAG."
        )
    state = longrun.run_status(RUN_DIR)["state"]
    if state == "done":
        break
    if state == "paused":
        print("paused: the time budget stopped the run after a saved epoch. Run all again in a new session to resume.")
        break
    if state == "failed":
        explain(f"failed: {(longrun.run_status(RUN_DIR)['lock'] or {}).get('error', 'see the log')}")
    if state in ("stale", "new") or is_dead():
        time.sleep(20)  # a finish writes DONE.json, then the lock; read both again before calling it dead
        state = longrun.run_status(RUN_DIR)["state"]
        if state == "done":
            break
        if state == "paused":
            print("paused: the time budget stopped the run after a saved epoch. Run all again in a new session to resume.")
            break
        if state in ("stale", "new", "failed") or is_dead():
            explain(f"is not running and not done (state {state})")
    quiet, limit = quiet_seconds(), stall_limit()
    if quiet > limit:
        print(f"\n{RUN_TAG} is alive but its log has not grown. End of its log:\n{tail(RUN_DIR / 'train.log')}")
        raise RuntimeError(
            f"{RUN_TAG} is stuck: no new line in train.log or metrics.jsonl for {quiet:.0f} s (limit {limit:.0f} s). "
            f"Its pid is in {RUN_DIR / gates.RUNNING_FILE}. Stop it, then Run all again"
        )
    time.sleep(POLL_SECONDS)

print(line())

## Leaderboard

Every run folder under `ROOT`, best validation score first: epoch reached (of its target), best val FG mIoU and the epoch it was reached, learning rate, seed, git sha, data fingerprint, parent, and per-class IoU at each run's best epoch. `state` is `running` (live heartbeat), `stale` (a lock whose heartbeat is older than 10 minutes, or a folder with no lock), `paused` (time budget), `done` or `failed`. A fork's epoch counts from its own start; the parent column says where it began. Runs with different data fingerprints are flagged: their scores are not comparable.

In [ ]:
ROWS = longrun.scan(ROOT)
print(longrun.leaderboard(ROWS))
print(f"\nroot {ROOT}  commit {COMMIT}")

## Plots

This run: train loss, and train and val FG mIoU against epoch (validation every `EVAL_EVERY` epochs). Then per-class val IoU at the best checkpoint of this run and of `COMPARE_TAGS` (put the CNN control arm there). Then every run's val curve together, a fork drawn after its parent's epoch.

In [ ]:
import matplotlib

if not IN_COLAB and not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sw.data import CLASS_NAMES


def curves(tag):
    rows = gates.read_metrics(ROOT / tag / "metrics.jsonl")
    return {
        "epoch": [r["epoch"] for r in rows],
        "loss": [r["train_loss"] for r in rows],
        "train": [r.get("train_fg_miou") for r in rows],
        "val": [r.get("val_fg_miou") for r in rows],
    }


def points(xs, ys):
    pairs = [(x, y) for x, y in zip(xs, ys) if y is not None]
    return [p[0] for p in pairs], [p[1] for p in pairs]


MINE = curves(RUN_TAG)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(MINE["epoch"], MINE["loss"])
left.set(xlabel="epoch", ylabel="train loss", title=f"{RUN_TAG}: train loss")
left.set_yscale("log")
right.plot(*points(MINE["epoch"], MINE["train"]), label="train FG mIoU")
right.plot(*points(MINE["epoch"], MINE["val"]), marker="o", ms=3, label="val FG mIoU")
for name, value in (("fused 0.286", 0.286), ("collapsed 0.298", 0.298), ("CNN 0.403", 0.403)):
    right.axhline(value, color="gray", ls=":", lw=0.8)
    right.text(MINE["epoch"][0], value, "30-epoch screen " + name, fontsize=7, va="bottom")
right.set(xlabel="epoch", ylabel="FG mIoU", title=f"{RUN_TAG}: train and val")
right.legend()
plt.tight_layout()
plt.show()

shown = [row for row in ROWS if row["tag"] in (RUN_TAG, *COMPARE_TAGS) and row["per_class"]]
if shown:
    width = 0.8 / len(shown)
    plt.figure(figsize=(10, 4))
    for i, row in enumerate(shown):
        values = [row["per_class"].get(name) or 0.0 for name in CLASS_NAMES]
        plt.bar([k + i * width for k in range(len(CLASS_NAMES))], values, width, label=f"{row['tag']} (best {row['best_val']:.3f} at {row['best_epoch']})")
    plt.xticks([k + 0.4 - width / 2 for k in range(len(CLASS_NAMES))], CLASS_NAMES, rotation=30)
    plt.ylabel("val IoU at the best checkpoint")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("no validated epoch yet for the per-class plot")

In [ ]:
plt.figure(figsize=(10, 5))
for row in ROWS:
    if row["epoch"] == 0:
        continue
    curve = curves(row["tag"])
    xs, ys = points([e + row["offset"] for e in curve["epoch"]], curve["val"])
    if xs:
        plt.plot(xs, ys, label=f"{row['tag']} ({row['owner']}, {row['state']})")
plt.xlabel("epoch, counted from the start of the first ancestor")
plt.ylabel("val FG mIoU")
plt.title("all runs under the shared root")
plt.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Soup (experimental)

**Experimental.** Averages the weights of the runs in `SOUP_RUNS`, which must all be forks of the same parent checkpoint (see their `metadata.json`), then scores the average and each ingredient on the full val split. The average is saved as a new run under `SOUP_TAG` (`best.pt`, `metadata.json`, `DONE.json`) **only if it scores above every ingredient**; otherwise nothing is written. Uniform averaging of weights from a shared start is a known trick; here it is not tested. Needs `SOUP_RUNS` (two or more tags) and the data on this VM. Skipped when `SOUP_RUNS` is empty.

In [ ]:
if not SOUP_RUNS:
    print("SOUP_RUNS is empty: skipped")
else:
    SOUP_TAG = SOUP_TAG or f"{RUNNER_NAME.lower()}-soup"
    SOUP = longrun.soup_runs(
        [ROOT / tag for tag in SOUP_RUNS],
        longrun.make_evaluator(ROOT / SOUP_RUNS[0], SW_ROOT, DEVICE),
        ROOT / SOUP_TAG,
        RUNNER_NAME,
    )
    for tag, score in SOUP["ingredients"].items():
        print(f"{tag:24s} val FG mIoU {score:.3f}")
    print(f"{'soup':24s} val FG mIoU {SOUP['soup_val_fg_miou']:.3f}")
    print(f"saved as {SOUP_TAG}" if SOUP["saved"] else "the soup did not beat every ingredient; nothing saved")

## What to send back

* The leaderboard above, and `ROOT/<RUN_TAG>/metrics.csv` and `metadata.json` of your runs.
* `docs/RESULTS.md` gets one row per finished run, with its flags, the epochs run (of the target), the git sha, and the owner.

These are val scores. Nothing here touches the test split. The official split only; no pretrained weights; the attention is `src/sw/attention.py`.